# Night-time lights — reproducible pipeline

**This notebook uses two kernels, bridged by Google Drive** — the same
pattern as xKDR's [Russia-Ukraine reproducible notebook](https://colab.research.google.com/github/xKDR/Shedding-light-on-the-Russia-Ukraine-war/blob/main/reproducible_research.ipynb).

1. **Part 1 (Python)** mounts Drive, then downloads the monthly VIIRS
   rasters from Google Earth Engine straight into a Drive folder via
   `getDownloadURL` (synchronous, parallelised — far faster than the GEE
   task queue at coarse scales).
2. You **change the Colab runtime to Julia** at the banner. Colab clears
   `/content` on a runtime switch — but **Google Drive is mounted storage,
   so the rasters survive**.
3. **Part 2 (Julia)** reads the rasters straight from the mounted Drive,
   cleans them with `NighttimeLights.clean_complete` (the PSTT2021
   pipeline), aggregates to districts, and writes `viirs_monthly.csv` back
   to the same Drive folder.

Output: `viirs_monthly.csv` in *My Drive / India-Built-and-Lit-VIIRS*.


---
# Part 1 · Python — download VIIRS rasters

Run with the default **Python 3** runtime.


## 1 · Mount Drive + Earth Engine

In [ ]:
!pip install -q -U earthengine-api

from google.colab import drive
drive.mount("/content/drive")          # <- the bridge across the kernel switch

import ee
ee.Authenticate()

PROJECT = "gee-ntl-470405"             # <-- your GEE Cloud project id
ee.Initialize(project=PROJECT)
print("Drive mounted, Earth Engine ready.")

## 2 · Parameters

`EXPORT_SCALE_M` is the **resampling knob**. VIIRS is 500 m native:
500 = native, 1000 = 1/2 each axis, 2000 = 1/4, 10000 = 1/20. The whole
cleaned India cube must fit in RAM in Part 2 — pick to suit. At ≥2000 m the
data per month is small enough that the synchronous fetch below is by far
the fastest option.


In [ ]:
START = (2014, 1)            # first (year, month), inclusive
END   = (2025, 12)           # last  (year, month), inclusive

EXPORT_SCALE_M = 1000        # resampling knob

DRIVE_FOLDER = "India-Built-and-Lit-VIIRS"
BOUNDARY_URL = "https://raw.githubusercontent.com/xKDR/India-Built-and-Lit/main/data/boundaries/districts_simplified.geojson"
VIIRS_COLL   = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"

## 3 · Export region (bounding box of all districts, from GitHub)

In [ ]:
import json, urllib.request

with urllib.request.urlopen(BOUNDARY_URL) as resp:
    gj = json.load(resp)
feats = [ee.Feature(ee.Geometry(ft["geometry"], proj="EPSG:4326", geodesic=False))
         for ft in gj["features"]]
region = ee.FeatureCollection(feats).geometry().bounds()
print("export region ready")

## 4 · Download one 2-band GeoTIFF per month → Drive

One `Export.image.toDrive` task per month — each writes a 2-band GeoTIFF
(band 1 `avg_rad`, band 2 `cf_cvg`; `clean_complete` needs both) to the
Drive folder. The cell queues all tasks first, then waits on them with a
single progress bar that prints each task's state every 30 s. Already-present
TIFs on Drive are skipped so the cell is resumable.

Heads-up: this is async. With GEE's typical 3-5 concurrent slots, ~144 monthly
exports take roughly 30-60 min wall time at coarse scales (most of it is
queue wait, not compute). You can also monitor at
https://code.earthengine.google.com/tasks.


In [ ]:
import os, time
from collections import Counter
from tqdm.auto import tqdm


def months(start, end):
    y, m = start
    while (y, m) <= end:
        yield y, m
        m += 1
        if m == 13:
            m, y = 1, y + 1


drive_dir = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(drive_dir, exist_ok=True)
have = set(os.listdir(drive_dir))

# Queue one Export.toDrive task per month not already on Drive.
tasks = {}
for y, m in months(START, END):
    desc = f"viirs_{y}_{m:02d}"
    if f"{desc}.tif" in have:
        continue
    start = ee.Date.fromYMD(y, m, 1)
    img = (ee.ImageCollection(VIIRS_COLL)
           .filterDate(start, start.advance(1, "month"))
           .select(["avg_rad", "cf_cvg"])
           .first())
    if img is None:
        print(f"  no image for {desc}, skipping")
        continue
    t = ee.batch.Export.image.toDrive(
        image=ee.Image(img).toFloat(), description=desc,
        folder=DRIVE_FOLDER, fileNamePrefix=desc, region=region,
        scale=EXPORT_SCALE_M, crs="EPSG:4326", maxPixels=1e13)
    t.start()
    tasks[desc] = t

print(f"{len(tasks)} task(s) queued.  {len(have)} already on Drive.")

# Wait for every task to reach a terminal state.
# The postfix shows live per-state counts (ready, running, succeeded, …) so
# progress is visible even before the first task completes.
TERM = ("COMPLETED", "SUCCEEDED", "FAILED", "CANCELLED")
done = set()
with tqdm(total=len(tasks), desc="exports") as bar:
    while len(done) < len(tasks):
        states = Counter()
        for desc, t in tasks.items():
            if desc in done:
                continue
            try:
                st = t.status()["state"]
            except Exception as e:
                bar.write(f"  {desc}: status() error — {type(e).__name__}: {e}")
                continue
            states[st.lower()] += 1
            if st in TERM:
                done.add(desc)
                bar.update(1)
                if st not in ("COMPLETED", "SUCCEEDED"):
                    bar.write(f"  ⚠ {desc}: {st}")
        bar.set_postfix(dict(states))     # live per-state counts
        if len(done) < len(tasks):
            time.sleep(30)

print(f"done -> {drive_dir}")

---
# ⚠️  SWITCH THE COLAB RUNTIME TO JULIA NOW

**Runtime → Change runtime type → "Julia" → choose version `1.11`**, then
**Connect**.

> **Important:** pick Julia **1.11**, not the default (currently 1.12).
> `NighttimeLights.jl` v0.x pins
> `DimensionalData = "0.27.3"` and `julia = "1.10"`; that DimensionalData
> version doesn't precompile under Julia 1.12 (the `UndefVarError: \`X\`
> not defined in DimensionalData.Dimensions` you'll otherwise see). 1.11
> works.

Colab clears `/content` on the switch, **but Google Drive is mounted
storage** — the VIIRS rasters Part 1 wrote are still in
*My Drive / `India-Built-and-Lit-VIIRS`*. Part 2 reads them straight from
there.

If Drive isn't visible after the switch, mount it from the Colab file
browser (folder icon in the sidebar → "Mount Drive").

---
# Part 2 · Julia — clean + aggregate


## 5 · Install Julia packages

In [ ]:
using Pkg

# ───── Fast-path: pre-built Julia depot from a GitHub release ─────
# Saves ~30 minutes of precompilation on first run. Skipped if the depot is
# already in place (re-runs are no-ops).
const DEPOT_URL = "https://github.com/xKDR/India-Built-and-Lit/releases/download/init/julia_depot.tar.gz"

if !isdir(expanduser("~/.julia/packages/Rasters"))
    println("downloading pre-built Julia depot (~1-2 GB)…")
    using Downloads
    Downloads.download(DEPOT_URL, "/tmp/depot.tar.gz")
    println("extracting…")
    run(`tar -xzf /tmp/depot.tar.gz -C $(expanduser("~/"))`)
    rm("/tmp/depot.tar.gz")
    println("depot ready.")
end

# ───── Packages (no-ops after the depot extract; otherwise full install) ─
Pkg.add(["Rasters", "ArchGDAL", "DataFrames", "CSV",
         "Dates", "Statistics", "Plots", "JSON"])
Pkg.add(url="https://github.com/xKDR/NighttimeLights.jl")

using Rasters, ArchGDAL, NighttimeLights
using DataFrames, CSV, Dates, Statistics, Plots, JSON
println("Julia packages ready")

## 6 · Point at the Drive folder

The Drive mount persists across kernels in a Colab session, so
`/content/drive/MyDrive/<folder>` is simply there after the runtime switch.


In [ ]:
const DRIVE_FOLDER = "India-Built-and-Lit-VIIRS"
const RASTER_DIR   = joinpath("/content/drive/MyDrive", DRIVE_FOLDER)
const OUT          = joinpath(RASTER_DIR, "viirs_monthly.csv")
const BG_THRESHOLD = 4
const BOUNDARY_URL = "https://raw.githubusercontent.com/xKDR/India-Built-and-Lit/main/data/boundaries/districts_simplified.geojson"
@assert isdir(RASTER_DIR) "Drive not visible - mount it via the file browser"
println("raster dir: ", RASTER_DIR)

## 7 · Load the monthly TIFs into datacubes

Each GeoTIFF has 2 bands — band 1 `avg_rad`, band 2 `cf_cvg`. Each slice is
materialised with `read()` so the combined cube is a plain in-memory array.


In [ ]:
# Pattern-match upfront so stray files (.gstmp, shortcuts, etc.) are skipped.
const VIIRS_RX = r"^viirs_(\d{4})_(\d{2})\.tif$"
files = sort(filter(f -> occursin(VIIRS_RX, f), readdir(RASTER_DIR)))
isempty(files) && error("no VIIRS TIFs in $RASTER_DIR - did Part 1 finish?")

function tdate(f)
    m = match(VIIRS_RX, f)
    Date(parse(Int, m[1]), parse(Int, m[2]))
end
dates = tdate.(files)

rad = [read(Raster(joinpath(RASTER_DIR, f); lazy=true))[Band=1] for f in files]
cf  = [read(Raster(joinpath(RASTER_DIR, f); lazy=true))[Band=2] for f in files]

rad_cube = Rasters.combine(RasterSeries(rad, Ti(dates)), Ti)
cf_cube  = Rasters.combine(RasterSeries(cf,  Ti(dates)), Ti)
println("rad cube: ", size(rad_cube), "  (", length(files), " months)")

## 8 · Clean with `NighttimeLights.clean_complete`

Runs the full PSTT2021 pipeline (NA recoding, background-noise masking,
stable-pixel detection, Hampel outlier removal, cloud-bias correction,
interpolation) over the whole India cube at once. `bgthreshold=4` suppresses
low-level rural noise that otherwise dominates district sums.


In [ ]:
cleaned = clean_complete(rad_cube, cf_cube; bgthreshold=BG_THRESHOLD)
println("cleaned cube: ", size(cleaned))

## 9 · Aggregate cleaned pixels to districts

In [ ]:
gjson_path = download(BOUNDARY_URL)      # SHRUG districts from GitHub

districts = NamedTuple[]
ArchGDAL.read(gjson_path) do ds
    layer = ArchGDAL.getlayer(ds, 0)
    for feat in layer
        geom = ArchGDAL.getgeom(feat)
        geom === nothing && continue
        push!(districts, (
            pc11_s_id = string(ArchGDAL.getfield(feat, "pc11_s_id")),
            pc11_d_id = string(ArchGDAL.getfield(feat, "pc11_d_id")),
            d_name    = string(ArchGDAL.getfield(feat, "d_name")),
            geom      = geom))
    end
end
println(length(districts), " districts")

ts = collect(dims(cleaned, Ti))
rows = NamedTuple[]
for (i, d) in enumerate(districts)
    i % 50 == 0 && println("  ", i, "/", length(districts))
    local masked
    try
        masked = Rasters.mask(Rasters.crop(cleaned; to=d.geom); with=d.geom)
    catch e
        @warn "skip $(d.pc11_d_id) ($(d.d_name)): $(sprint(showerror, e))"
        continue
    end
    for t in ts
        sl = view(masked, Ti=At(t))
        n  = count(!ismissing, sl)
        s  = n == 0 ? missing : sum(skipmissing(sl))
        push!(rows, (pc11_s_id=d.pc11_s_id, pc11_d_id=d.pc11_d_id,
                     d_name=d.d_name, year=year(t), month=month(t),
                     date=Date(t),
                     sum_radiance = s,
                     mean_radiance = n == 0 ? missing : s / n,
                     n_pixels = n))
    end
end
panel = DataFrame(rows)
sort!(panel, [:pc11_s_id, :pc11_d_id, :year, :month])
nrow(panel)

## 10 · Write `viirs_monthly.csv` to Drive

In [ ]:
CSV.write(OUT, panel)
println("wrote ", nrow(panel), " rows -> ", OUT)
println("(it's in My Drive / ", DRIVE_FOLDER, ")")
first(panel, 8)

## 11 · Plot — national monthly night-time lights

In [ ]:
agg = DataFrames.combine(groupby(dropmissing(panel, :sum_radiance), :date),
               :sum_radiance => sum => :total)
sort!(agg, :date)

plot(agg.date, agg.total,
     title  = "India — total VIIRS night-time lights by month",
     xlabel = "month",
     ylabel = "sum radiance (nW cm⁻² sr⁻¹)",
     label  = false,
     linewidth = 2,
     size   = (900, 600))

## 12 · Top 20 districts by NTL — latest year

In [ ]:
# Last year with all 12 months present (skip current partial year).
months_per_year = DataFrames.combine(
    groupby(dropmissing(panel, :sum_radiance), :year),
    :month => (m -> length(unique(m))) => :n)
full = months_per_year.year[months_per_year.n .>= 12]
yr = isempty(full) ? maximum(months_per_year.year) : maximum(full)

year_rows = filter(r -> year(r.date) == yr && r.sum_radiance !== missing, panel)
top = DataFrames.combine(
    groupby(year_rows, [:pc11_s_id, :pc11_d_id, :d_name]),
    :sum_radiance => sum => :total)
sort!(top, :total)               # ascending — largest ends at top of the chart
top20 = last(top, 20)
top20.state = state_name.(top20.pc11_s_id)

# Draw bars as Shape rectangles ourselves — Plots.jl's bar(...; orientation=:h)
# is broken in some Plots versions; this idiom works on every backend.
gr()
n = nrow(top20)
fig = plot(
    legend      = false,
    xlabel      = "NTL sum radiance (nW cm⁻² sr⁻¹)",
    title       = "Top 20 districts — $yr",
    yticks      = (1:n, top20.d_name),
    ylims       = (0.4, n + 0.6),
    xlims       = (0, maximum(top20.total) * 1.05),
    size        = (1100, 700),
    left_margin = 15Plots.mm)

for i in 1:n
    v = top20.total[i]
    plot!(fig, Shape([0, v, v, 0], [i - 0.4, i - 0.4, i + 0.4, i + 0.4]);
          color = "#e07a5f", linecolor = "#e07a5f", label = "")
    annotate!(fig, v * 0.5, i, Plots.text(top20.state[i], 9, :center, :black))
end
fig

## 13 · Monthly NTL trajectory — selected districts

One line per district. Edit `cities` to taste.

In [ ]:
cities = ["Mumbai", "New Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad"]

sub = filter(r -> r.d_name in cities && r.sum_radiance !== missing, panel)
gdf = groupby(sub, :d_name)

fig = plot(; xlabel = "Month",
             ylabel = "Sum radiance (nW cm⁻² sr⁻¹)",
             title  = "Monthly VIIRS NTL — selected districts",
             size   = (900, 420), legend = :topleft)
for g in gdf
    plot!(fig, g.date, g.sum_radiance; label = first(g.d_name), lw = 1.5)
end
fig

## 14 · Choropleth — NTL by district, latest year

Two flavours, same data:

- A **static** version using the default `gr` Plots backend (good for
  exports and PDFs).
- An **interactive** version using `Plots.plotly()` (zoom, pan, hover —
  the same renderer the web dashboard uses). Plots' plotly backend needs
  `PlotlyBase` installed; the cell between handles that.

In [ ]:
# District totals for the latest year
yr = maximum(year.(panel.date))
year_rows = filter(r -> year(r.date) == yr && r.sum_radiance !== missing, panel)
totals = DataFrames.combine(groupby(year_rows, :pc11_d_id),
                 :sum_radiance => sum => :total)
val_of = Dict(string(k) => v for (k, v) in zip(totals.pc11_d_id, totals.total))

# Log-scale colour mapping (NTL totals span several orders of magnitude)
pos = filter(>(0), collect(values(val_of)))
lo, hi = log10(minimum(pos)), log10(maximum(pos))
cmap   = cgrad(:OrRd)
shade(v) = (v === nothing || v <= 0) ? RGBA(0.88, 0.88, 0.88, 1.0) :
           cmap[clamp((log10(v) - lo) / (hi - lo), 0.0, 1.0)]

# Iterate features, draw one Shape per polygon (static, gr backend)
gj = JSON.parsefile(gjson_path)

fig = plot(; aspect_ratio = :equal, legend = false, framestyle = :none,
             title = "VIIRS night-time lights by district — $yr",
             size = (900, 900))

for feat in gj["features"]
    d_id  = string(feat["properties"]["pc11_d_id"])
    color = shade(get(val_of, d_id, nothing))
    geom  = feat["geometry"]
    polys = geom["type"] == "MultiPolygon" ? geom["coordinates"]      :
            geom["type"] == "Polygon"      ? [geom["coordinates"]]    : Any[]
    for poly in polys
        outer = poly[1]
        xs = Float64[pt[1] for pt in outer]
        ys = Float64[pt[2] for pt in outer]
        plot!(fig, Shape(xs, ys);
              fillcolor = color, linecolor = :black, linewidth = 0.15)
    end
end
fig

In [ ]:
Pkg.add("PlotlyBase")

In [ ]:
plotly()                # interactive Plotly backend, built into Plots.jl

# District totals for the latest year
yr = maximum(year.(panel.date))
year_rows = filter(r -> year(r.date) == yr && r.sum_radiance !== missing, panel)
totals = DataFrames.combine(groupby(year_rows, [:pc11_d_id, :d_name]),
                 :sum_radiance => sum => :total)
val_of  = Dict(string(k) => v for (k, v) in zip(totals.pc11_d_id, totals.total))
name_of = Dict(string(k) => v for (k, v) in zip(totals.pc11_d_id, totals.d_name))

# Log-scale colour mapping (NTL totals span several orders of magnitude)
pos    = filter(>(0), collect(values(val_of)))
lo, hi = log10(minimum(pos)), log10(maximum(pos))
cmap   = cgrad(:OrRd)
shade(v) = (v === nothing || v <= 0) ? RGBA(0.88, 0.88, 0.88, 1.0) :
           cmap[clamp((log10(v) - lo) / (hi - lo), 0.0, 1.0)]

# Draw one Plots.Shape per polygon. JSON.parsefile gives nested arrays
# that we hand straight to Shape(xs, ys).
gj = JSON.parsefile(gjson_path)

fig = plot(; aspect_ratio = :equal, legend = false, framestyle = :none,
             title = "VIIRS night-time lights by district — $yr",
             size = (900, 900))

for feat in gj["features"]
    d_id  = string(feat["properties"]["pc11_d_id"])
    color = shade(get(val_of, d_id, nothing))
    label = get(name_of, d_id, d_id)
    geom  = feat["geometry"]
    polys = geom["type"] == "MultiPolygon" ? geom["coordinates"]   :
            geom["type"] == "Polygon"      ? [geom["coordinates"]] : Any[]
    for poly in polys
        outer = poly[1]
        xs = Float64[pt[1] for pt in outer]
        ys = Float64[pt[2] for pt in outer]
        plot!(fig, Shape(xs, ys);
              fillcolor = color, linecolor = :black, linewidth = 0.15,
              hover = label)
    end
end
fig

## 15 · Monthly NTL by state (trend)

One line per PC11 state, summed across that state's districts. Mirrors the
"Trend by state" chart on the dashboard.

In [ ]:
const PC11_STATES = Dict(
    "1"=>"Jammu & Kashmir", "2"=>"Himachal Pradesh", "3"=>"Punjab",
    "4"=>"Chandigarh",      "5"=>"Uttarakhand",      "6"=>"Haryana",
    "7"=>"NCT of Delhi",    "8"=>"Rajasthan",        "9"=>"Uttar Pradesh",
    "10"=>"Bihar",          "11"=>"Sikkim",          "12"=>"Arunachal Pradesh",
    "13"=>"Nagaland",       "14"=>"Manipur",         "15"=>"Mizoram",
    "16"=>"Tripura",        "17"=>"Meghalaya",       "18"=>"Assam",
    "19"=>"West Bengal",    "20"=>"Jharkhand",       "21"=>"Odisha",
    "22"=>"Chhattisgarh",   "23"=>"Madhya Pradesh",  "24"=>"Gujarat",
    "25"=>"Daman & Diu",    "26"=>"Dadra & Nagar Haveli",
    "27"=>"Maharashtra",    "28"=>"Andhra Pradesh",  "29"=>"Karnataka",
    "30"=>"Goa",            "31"=>"Lakshadweep",     "32"=>"Kerala",
    "33"=>"Tamil Nadu",     "34"=>"Puducherry",      "35"=>"Andaman & Nicobar Is.",
)
state_name(sid) = get(PC11_STATES, string(parse(Int, string(sid))), "state $sid")

state_dt = DataFrames.combine(
    groupby(dropmissing(panel, :sum_radiance), [:pc11_s_id, :date]),
    :sum_radiance => sum => :total)
sort!(state_dt, [:pc11_s_id, :date])

gr()                                    # back to the static backend for this one

fig = plot(; xlabel = "Month",
             ylabel = "Sum radiance (nW cm⁻² sr⁻¹)",
             title  = "Monthly VIIRS NTL — sum by state",
             size   = (1000, 560),
             legend = :outerright, legendfontsize = 7,
             foreground_color_legend = nothing)
for g in groupby(state_dt, :pc11_s_id)
    plot!(fig, g.date, g.total;
          label = state_name(first(g.pc11_s_id)), lw = 1)
end
fig

## 16 · Built-up volume vs NTL (latest common year)

Loads `bv_annual.csv` from the BV Drive folder (produced by
`building_volume.ipynb`) and plots each district as one point — log-log,
NTL on the y-axis, built-up volume on the x-axis. Same view as the
"Building volume vs. NTL" scatter on the dashboard.

In [ ]:
const BV_CSV_URL = "https://raw.githubusercontent.com/xKDR/India-Built-and-Lit/main/docs/data/bv_annual.csv"

bv_path = download(BV_CSV_URL)            # → local temp path
bv = CSV.read(bv_path, DataFrame)
bv.pc11_d_id = string.(bv.pc11_d_id)
bv.year = Int.(bv.year)

# Annual NTL per district
annual_ntl = DataFrames.combine(
    groupby(dropmissing(panel, :sum_radiance), [:pc11_d_id, :year]),
    :sum_radiance => sum => :ntl)
annual_ntl.pc11_d_id = string.(annual_ntl.pc11_d_id)
annual_ntl.year = Int.(annual_ntl.year)

joined = innerjoin(bv, annual_ntl, on = [:pc11_d_id, :year])
yr = maximum(joined.year)
sub = filter(r -> r.year == yr && r.volume_m3 > 0 && r.ntl > 0, joined)

gr()
scatter(sub.volume_m3, sub.ntl;
        xscale = :log10, yscale = :log10,
        xlabel = "Built-up volume (m³)",
        ylabel = "NTL sum radiance",
        title  = "Built-up volume vs night-time lights — $yr",
        markersize = 3, markeralpha = 0.5,
        color = :orange, legend = false,
        size = (800, 540))